<a href="https://colab.research.google.com/github/oliversebastianmartinesdiaz-cmyk/Applied-Artificial-Intelligence-Course-with-Llama/blob/main/Challenge_2_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq --quiet

from groq import Groq

from google.colab import userdata

client = Groq(api_key=userdata.get('API_GROQ2'))

print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [ ]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."
response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto


In [ ]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."

Sentimiento: Negativo

Reseña: "Nunca llegó mi pedido, pésimo servicio."

Sentimiento: Mixto

Reseña: "El envío llegó tarde pero el producto es excelente."

Sentimiento:"""

response_few = client.chat.completions.create(

    model="openai/gpt-oss-20b",

    messages=[{"role": "user", "content": prompt_few_shot}]

)

print("Few-shot:", response_few.choices[0].message.content)

Few-shot: **Clasificación corregida:**

| Reseña | Sentimiento correcto |
|--------|----------------------|
| "Me encantó, llegó rápido y en perfecto estado." | **Positivo** |
| "Nunca llegó mi pedido, pésimo servicio." | **Negativo** |
| "El envío llegó tarde pero el producto es excelente." | **Mixto** |

**Explicación:**

1. La primera reseña expresa satisfacción total con el producto y la entrega, por lo tanto es **Positivo**.  
2. La segunda reseña muestra descontento con la entrega y el servicio, lo que la clasifica como **Negativo**.  
3. La tercera reseña combina un aspecto negativo (el envío llegó tarde) y un aspecto positivo (el producto es excelente), resultando en una clasificación **Mixto**.


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [ ]:
# Razonamiento paso a paso (chain-of-thought)
problema = (

    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "

    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "

    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."

)

response_cot = client.chat.completions.create(

    model="openai/gpt-oss-20b",

    messages=[{"role": "user", "content": problema}]

)

print(response_cot.choices[0].message.content)

**Paso 1 – Determinar la distancia adelantada por el primer tren**  
El primer tren parte con 80 km/h. Cuando el segundo tren sale, ya han transcurrido 2 h, por lo que:

\[
\text{Distancia recorrida por el primero} = 80\ \text{km/h} \times 2\ \text{h} = 160\ \text{km}
\]

**Paso 2 – Calcular la velocidad relativa entre los trenes**  
El segundo tren viaja a 120 km/h y el primero a 80 km/h. La velocidad con la que el segundo tren cierra la brecha es:

\[
\text{Velocidad relativa} = 120\ \text{km/h} - 80\ \text{km/h} = 40\ \text{km/h}
\]

**Paso 3 – Encontrar el tiempo de persecución**  
Para cubrir los 160 km de ventaja, el tiempo requerido es:

\[
t = \frac{\text{Distancia}}{\text{Velocidad relativa}} = \frac{160\ \text{km}}{40\ \text{km/h}} = 4\ \text{h}
\]

**Respuesta**  
El segundo tren tarda **4 horas** en alcanzar al primero (lo cual equivale a 6 horas desde la salida del primero).


### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [ ]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (

    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "

    "2026? Respuesta muy breve y corta."

)

response_alucinacion = client.chat.completions.create(

    model="openai/gpt-oss-20b",

    messages=[{"role": "user", "content": prompt_desconocido}]

)

print(response_alucinacion.choices[0].message.content)

Equipo 42 ganó la final.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [ ]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer

import numpy as np

In [ ]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [

    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "

    "empaque original.",

    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "

    "envío de regreso.",

    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "

    "cambio de talla."

]

embeddings_documentos = modelo_embeddings.encode(documentos)

print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [ ]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):

    embedding_pregunta = modelo_embeddings.encode([pregunta])

    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()

    indice_mas_similar = np.argmax(similitudes)

    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"

fragmento = buscar_fragmento(pregunta)

print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [ ]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(

    model="openai/gpt-oss-20b",

    messages=[{"role": "user", "content": prompt_rag}]

)

print(response_rag.choices[0].message.content)

NameError: name 'client' is not defined

# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [ ]:
# Leer API key, instalar e importar librerías

!pip install groq --quiet
from groq import Groq
from google.colab import userdata
client = Groq(api_key=userdata.get('API_GROQ2'))
print("Cliente de Groq inicializado correctamente.")

!pip install sentence-transformers --quiet
from sentence_transformers import SentenceTransformer
import numpy as np


Cliente de Groq inicializado correctamente.


In [ ]:
# Definir la lista documentos y generar sus embeddings
#Siguiendo el regalmento de competencia de SUMO de ROBOMATRIX con fecha de 14 de enero, 2022

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
documentos = [
    "En la categoría de Mega Sumo, el peso máximo del robot no debe exceder"
    "los 3.0 kilogramos y sus dimensiones iniciales no deben superar 20 x 20 cm.",
    "El robot debe iniciar su movimiento de combate exactamente 5 segundos"
    "después de presionar el botón de arranque.",
    "Un robot pierde el round si cualquier parte de su estructura"
    " toca el exterior del dohyo (la zona fuera de la línea blanca)."
]
embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [ ]:
# Definir la función buscar_fragmento
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [ ]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
pregunta = "¿Cuánto tiempo debe esperar el robot de sumo antes de moverse tras presionar el botón?"
prompt_sin_rag = f"Responde la siguiente pregunta sobre el torneo de Sumo Robomatrix: {pregunta}"
response_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",  # o el modelo de Llama disponible en tu Groq
    messages=[{"role": "user", "content": prompt_sin_rag}]
)
respuesta_sin_rag = response_sin_rag.choices[0].message.content

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [ ]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
fragmento_recuperado = buscar_fragmento(pregunta)
prompt_con_rag = f"""Responde la pregunta usando ÚNICAMENTE el siguiente fragmento del reglamento.
Reglamento: {fragmento_recuperado}
Pregunta: {pregunta}
Respuesta:"""
response_con_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_con_rag}]
)
respuesta_con_rag = response_con_rag.choices[0].message.content

**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [ ]:
# Mostrar ambas respuestas para comparar
print("--- PREGUNTA ---")
print(pregunta)
print("\n--- RESPUESTA SIN RAG (Posible alucinación) ---")
print(respuesta_sin_rag)
print("\n--- RESPUESTA CON RAG (Información exacta del reglamento) ---")
print(respuesta_con_rag)

--- PREGUNTA ---
¿Cuánto tiempo debe esperar el robot de sumo antes de moverse tras presionar el botón?

--- RESPUESTA SIN RAG (Posible alucinación) ---
En las reglas oficiales del torneo **Sumo Robomatrix** se establece que, tras presionar el botón de inicio, el robot **debe esperar 5 segundos** antes de poder comenzar a moverse.  

Esta espera de 5 s sirve para:

1. **Garantizar la seguridad** de los participantes y del público.  
2. **Permitir al árbitro** dar el “Go” definitivo antes de que el robot comience a actuar.  
3. **Uniformizar** el tiempo de inicio entre todos los equipos, evitando ventajas por reacciones demasiado rápidas.

Así que, en tu diseño y programación, implementa un retardo de **exactamente 5 segundos** entre la señal de pulsación del botón y el encendido de los motores.

--- RESPUESTA CON RAG (Información exacta del reglamento) ---
5 segundos.
